# 5강: 종합 실습 — SeekSeek 핵심 기술 통합

이 노트북에서는 SeekSeek의 핵심 기술을 조합하여
"MFT 스캔 → 인덱싱 → 검색" 전체 파이프라인을 직접 실행해봅니다.

> **⚠️ 주의**: 관리자 권한으로 실행해야 합니다.

> **버전 참고 (v1.0.1+)**
- `MAX_CONTENT_SIZE` 기본값은 `200MB`로 유지됩니다.
- 본문 색인은 워커 `2~4`개 + 소배치 즉시 flush 전략으로 메모리 피크를 완화합니다.
- 전체 색인 후 변경분 색인을 직렬로 이어서 수행합니다.

## 5.1 환경 설정

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(name)s: %(message)s')

# SeekSeek 모듈 임포트
from core.mft_scanner import get_ntfs_drives, enumerate_mft
from core.indexer import init_db, get_connection, upsert_file, upsert_content, get_stats
from core.searcher import search, match_label
from core.extractor import extract_text
from core import mft_cache
import config

print(f'DB 경로: {config.DB_PATH}')
print(f'NTFS 드라이브: {get_ntfs_drives()}')

## 5.2 DB 초기화

SeekSeek의 DB 스키마를 생성합니다.
- files + fts_files (FTS5 external content)
- file_contents + fts_contents (FTS5 external content)
- 트리거로 자동 동기화

In [ ]:
init_db()

with get_connection() as conn:
    stats = get_stats(conn)
    print(f'현재 DB 상태:')
    print(f'  총 파일 수:     {stats["total_files"]:,}')
    print(f'  내용 인덱싱 수: {stats["indexed_contents"]:,}')
    
    # 테이블 목록 확인
    tables = conn.execute(
        "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'trigger') ORDER BY type, name"
    ).fetchall()
    print(f'\nDB 오브젝트:')
    for name, typ in tables:
        print(f'  [{typ:7s}] {name}')

## 5.3 MFT 스캔 → 파일 캐시 구축

In [ ]:
import time

drive = 'C'  # 스캔할 드라이브

print(f'{drive}:\\ MFT 열거 시작...')
t0 = time.perf_counter()

result = enumerate_mft(drive)

elapsed = time.perf_counter() - t0
print(f'완료: {elapsed:.2f}초')
print(f'성공: {result.success}')

if result.success:
    print(f'파일/디렉터리 수: {result.total_entries:,}')
    print(f'Journal ID: {result.journal_id}')
    print(f'Next USN: {result.next_usn}')
    
    # 인메모리 캐시에 로드
    mft_cache.populate(result.files)
    print(f'캐시 로드 완료: {mft_cache.count():,}개')
else:
    print(f'오류: {result.error}')

## 5.4 파일명 검색 (MFT 캐시 모드)

In [ ]:
# MFT 캐시로 파일명 검색 (content_query 없음 → 캐시 모드)
queries = ['*.py', 'readme', 'main.py']

for q in queries:
    t0 = time.perf_counter()
    results = search(filename_query=q, max_results=10)
    elapsed = (time.perf_counter() - t0) * 1000
    
    print(f'\n=== "{q}" ({elapsed:.1f}ms, {len(results)}건) ===')
    for r in results[:5]:
        size_kb = r.size / 1024
        print(f'  {r.name:30s} {size_kb:>8.1f} KB  {r.path}')

## 5.5 특정 폴더 파일 DB 등록 + 텍스트 추출

In [ ]:
# 프로젝트 폴더의 Python 파일을 DB에 등록하고 텍스트를 추출
project_root = os.path.abspath('..')
indexed = 0

with get_connection() as conn:
    for root, dirs, files in os.walk(project_root):
        # .venv, __pycache__ 등 제외
        dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
        
        for name in files:
            ext = os.path.splitext(name)[1].lower()
            if ext not in ('.py', '.md', '.txt'):
                continue
            
            filepath = os.path.join(root, name)
            file_id = upsert_file(conn, filepath)
            
            if file_id is not None:
                text = extract_text(filepath)
                if text:
                    upsert_content(conn, file_id, text)
                    indexed += 1
    
    conn.commit()

print(f'DB 등록 완료: {indexed}개 파일 텍스트 색인')

with get_connection() as conn:
    stats = get_stats(conn)
    print(f'DB 상태: 총 {stats["total_files"]}개, 내용 {stats["indexed_contents"]}개')

## 5.6 FTS5 본문 검색

In [ ]:
# 본문 검색 (content_query 있음 → DB FTS5 모드)
content_queries = ['FTS5', 'MFT', 'extract_text']

for q in content_queries:
    t0 = time.perf_counter()
    results = search(content_query=q, max_results=10)
    elapsed = (time.perf_counter() - t0) * 1000
    
    print(f'\n=== 본문 "{q}" ({elapsed:.1f}ms, {len(results)}건) ===')
    for r in results[:5]:
        print(f'  [{match_label(r.match_type):6s}] {r.name:30s} {r.path}')

## 5.7 파일명 + 본문 복합 검색

In [ ]:
# 파일명과 본문을 동시에 검색하면 두 결과의 교집합(match_type='both')만 반환
results = search(
    filename_query='*.py',
    content_query='import',
    max_results=10,
)

print(f'파일명="*.py" AND 본문="import" → {len(results)}건')
for r in results:
    print(f'  [{match_label(r.match_type)}] {r.name}')

## 5.8 성능 비교: 캐시 검색 vs DB 검색

In [ ]:
import statistics

def benchmark(fn, label: str, repeat: int = 10):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = fn()
        times.append((time.perf_counter() - t0) * 1000)
    avg = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0
    print(f'{label:30s} avg={avg:.2f}ms ±{std:.2f}ms  결과={len(result)}건')

print('=== 성능 벤치마크 (10회 반복 평균) ===')
benchmark(lambda: search(filename_query='*.py'), 'MFT 캐시: *.py')
benchmark(lambda: search(filename_query='main'), 'MFT 캐시: main')
benchmark(lambda: search(content_query='import'), 'FTS5 DB: import')
benchmark(lambda: search(content_query='FTS5'), 'FTS5 DB: FTS5')

## 요약 — SeekSeek 파이프라인

```
1. MFT 열거 (enumerate_mft)
   → 전체 파일 목록 수집 (~수초)

2. 인메모리 캐시 (mft_cache.populate)
   → 파일명 검색 O(N) 순차 매칭 (<50ms)

3. DB 인덱싱 (upsert_file + upsert_content)
   → FTS5 역색인 트리거 자동 동기화

4. 검색 (search)
   → 파일명만: 캐시 모드
   → 본문 포함: FTS5 MATCH + BM25 랭킹
   → 복합: 교집합 (match_type='both')

5. USN 증분 업데이트 (별도 스레드)
   → 5초 폴링 → 캐시만 갱신
```